# Bài 2.7 · Gradient boosting trên dữ liệu thật

Bộ dữ liệu **Adult** (UCI, điều tra dân số Mỹ 1994) có gần 49 000 người, mỗi người 14 cột: tuổi, học vấn, nghề, số giờ làm mỗi tuần… Nhiệm vụ là đoán thu nhập năm có vượt 50 000 USD không. Metric là **ROC-AUC**, vì lớp dương chỉ chiếm khoảng một phần tư.

Notebook đi đúng khuôn của bài: chia test riêng, baseline trước, rồi LightGBM trong vòng 5-fold có early stopping, so learning rate với số cây, đọc độ quan trọng đặc trưng, cuối cùng so với XGBoost, CatBoost và trung bình ba mô hình. Chạy được trên CPU, khoảng 3 phút.

In [ ]:
import time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import lightgbm as lgb, xgboost as xgb, catboost, sklearn
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings('ignore')
SEED = 0
print('lightgbm', lgb.__version__, '| xgboost', xgb.__version__, '| catboost', catboost.__version__, '| sklearn', sklearn.__version__)

## 1. Đọc dữ liệu

`fetch_openml` tải bảng từ OpenML (cần bật Internet trong *Session options*). Cột chữ được giữ kiểu `category` để LightGBM, XGBoost và CatBoost chia trực tiếp theo nhóm, không cần one-hot.

In [ ]:
adult = fetch_openml('adult', version=2, as_frame=True)
X = adult.data.copy()
y = (adult.target == '>50K').astype(int).to_numpy()
cat_cols = X.select_dtypes(['category', 'object']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
for c in cat_cols:
    X[c] = X[c].astype('category')
print(X.shape, '| tỉ lệ >50K:', round(y.mean(), 3))
print('cột chữ:', cat_cols)
X.head()

## 2. Chia test riêng và dựng khuôn 5-fold

20% dữ liệu để riêng làm test, chỉ chấm một lần ở cuối. Mọi lựa chọn (mô hình, learning rate, số cây) dựa trên điểm **out-of-fold** của 5 fold trong phần train. `StratifiedKFold` giữ tỉ lệ lớp dương ở mọi fold.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_tr, y_tr))
print('train', X_tr.shape, '| test', X_te.shape)

## 3. Baseline

Đoán cùng một xác suất cho mọi người cho AUC đúng 0,5: mô hình nào cũng phải vượt mốc này. Baseline mạnh hơn là hồi quy logistic trên one-hot của cột chữ và cột số đã chuẩn hóa.

In [ ]:
print('hằng số:', roc_auc_score(y_te, np.full(len(y_te), y_tr.mean())))
pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
                         ('num', StandardScaler(), num_cols)])
logit = make_pipeline(pre, LogisticRegression(max_iter=2000))
X_tr_lr = X_tr.copy(); X_tr_lr[cat_cols] = X_tr_lr[cat_cols].astype(str)
auc_lr = cross_val_score(logit, X_tr_lr, y_tr, cv=folds, scoring='roc_auc')
print('logistic, AUC 5 fold:', auc_lr.round(4), '→ trung bình', auc_lr.mean().round(4))

## 4. LightGBM trong vòng 5-fold, có early stopping

Mỗi fold học trên 4 phần, dừng sớm khi AUC trên phần còn lại không tăng sau 100 vòng. Dự đoán của fold ghi đúng vào vị trí các hàng của nó, nên cuối cùng có một vector **OOF** phủ toàn bộ train để chấm một lần. Dự đoán test là trung bình của 5 mô hình.

In [ ]:
def run_lgb(params, rounds=5000, verbose=True):
    oof = np.zeros(len(y_tr)); pred = np.zeros(len(y_te)); iters = []
    for k, (a, b) in enumerate(folds):
        m = lgb.LGBMClassifier(n_estimators=rounds, random_state=SEED, verbose=-1, **params)
        m.fit(X_tr.iloc[a], y_tr[a], eval_set=[(X_tr.iloc[b], y_tr[b])], eval_metric='auc',
              callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = m.predict_proba(X_tr.iloc[b])[:, 1]
        pred += m.predict_proba(X_te)[:, 1] / len(folds)
        iters.append(m.best_iteration_)
        if verbose:
            print(f'fold {k}: AUC {roc_auc_score(y_tr[b], oof[b]):.4f}, dừng ở cây {m.best_iteration_}')
    return oof, pred, iters, m

t = time.time()
oof_lgb, pred_lgb, it_lgb, last_lgb = run_lgb(dict(learning_rate=0.05, num_leaves=31, subsample=0.8,
                                                   subsample_freq=1, colsample_bytree=0.8))
print(f'OOF AUC {roc_auc_score(y_tr, oof_lgb):.4f} | test AUC {roc_auc_score(y_te, pred_lgb):.4f} | {time.time() - t:.0f} giây')

## 5. Learning rate và số cây

Learning rate nhỏ thì mỗi cây chỉ sửa một phần nhỏ phần dư, nên cần nhiều cây hơn. Bảng dưới cho thấy đánh đổi này trên chính dữ liệu thật: số cây ở điểm dừng tăng gần tỉ lệ nghịch với learning rate, còn AUC chỉ nhích nhẹ.

In [ ]:
rows = []
for lr in [0.3, 0.1, 0.03]:
    t = time.time()
    oof, _, iters, _ = run_lgb(dict(learning_rate=lr, num_leaves=31, subsample=0.8, subsample_freq=1,
                                     colsample_bytree=0.8), verbose=False)
    rows.append({'learning_rate': lr, 'số cây (trung bình)': int(np.mean(iters)),
                 'OOF AUC': round(roc_auc_score(y_tr, oof), 4), 'giây': round(time.time() - t, 1)})
pd.DataFrame(rows)

## 6. Mô hình dựa vào cột nào?

Độ quan trọng theo **gain** là tổng độ giảm loss mà các phép chia trên một cột mang lại. Nó cho biết mô hình dùng cột nào nhiều, không chứng minh quan hệ nhân quả.

In [ ]:
imp = pd.Series(last_lgb.booster_.feature_importance('gain'), index=X_tr.columns).sort_values()
imp.tail(10).plot.barh(figsize=(7, 4), color='#c2603e')
plt.title('LightGBM: độ quan trọng theo gain (fold cuối)'); plt.tight_layout(); plt.show()

## 7. XGBoost, CatBoost và trung bình ba mô hình

Hai thư viện còn lại chạy trong cùng khuôn fold và cùng early stopping. Ba mô hình sai theo những cách hơi khác nhau, nên trung bình xác suất thường nhỉnh hơn từng mô hình riêng.

In [ ]:
def run_xgb():
    oof = np.zeros(len(y_tr)); pred = np.zeros(len(y_te))
    for a, b in folds:
        m = xgb.XGBClassifier(n_estimators=5000, learning_rate=0.05, max_depth=6, subsample=0.8,
                              colsample_bytree=0.8, tree_method='hist', enable_categorical=True,
                              eval_metric='auc', early_stopping_rounds=100, random_state=SEED)
        m.fit(X_tr.iloc[a], y_tr[a], eval_set=[(X_tr.iloc[b], y_tr[b])], verbose=False)
        oof[b] = m.predict_proba(X_tr.iloc[b])[:, 1]
        pred += m.predict_proba(X_te)[:, 1] / len(folds)
    return oof, pred

def run_cat():
    Xc_tr = X_tr.copy(); Xc_te = X_te.copy()
    for c in cat_cols:
        Xc_tr[c] = Xc_tr[c].astype(str); Xc_te[c] = Xc_te[c].astype(str)
    oof = np.zeros(len(y_tr)); pred = np.zeros(len(y_te))
    for a, b in folds:
        m = catboost.CatBoostClassifier(iterations=3000, learning_rate=0.08, depth=6, eval_metric='AUC',
                                        cat_features=cat_cols, random_seed=SEED, verbose=False,
                                        early_stopping_rounds=100)
        m.fit(Xc_tr.iloc[a], y_tr[a], eval_set=(Xc_tr.iloc[b], y_tr[b]))
        oof[b] = m.predict_proba(Xc_tr.iloc[b])[:, 1]
        pred += m.predict_proba(Xc_te)[:, 1] / len(folds)
    return oof, pred

t = time.time(); oof_xgb, pred_xgb = run_xgb(); print(f'XGBoost xong sau {time.time() - t:.0f} giây')
t = time.time(); oof_cat, pred_cat = run_cat(); print(f'CatBoost xong sau {time.time() - t:.0f} giây')
res = pd.DataFrame({
    'mô hình': ['logistic (baseline)', 'LightGBM', 'XGBoost', 'CatBoost', 'trung bình ba mô hình'],
    'OOF AUC': [auc_lr.mean(), roc_auc_score(y_tr, oof_lgb), roc_auc_score(y_tr, oof_xgb),
                roc_auc_score(y_tr, oof_cat), roc_auc_score(y_tr, (oof_lgb + oof_xgb + oof_cat) / 3)],
    'test AUC': [np.nan, roc_auc_score(y_te, pred_lgb), roc_auc_score(y_te, pred_xgb),
                 roc_auc_score(y_te, pred_cat), roc_auc_score(y_te, (pred_lgb + pred_xgb + pred_cat) / 3)]})
res.round(4)

## Đọc kết quả

Số dưới đây là của lần chạy mẫu; chạy lại có thể lệch ở chữ số thứ tư.

- Logistic đạt AUC khoảng 0,907; LightGBM, XGBoost và CatBoost cùng lên khoảng 0,929. Điểm OOF và điểm test gần như trùng nhau: khuôn 5-fold ước lượng đúng điểm trên dữ liệu chưa thấy.
- Learning rate 0,3 dừng ở khoảng 23 cây, 0,1 ở khoảng 90 cây, 0,03 ở khoảng 370 cây, trong khi AUC chỉ tăng từ 0,927 lên 0,929. Trong phòng thi, chọn learning rate theo thời gian còn lại.
- Trung bình ba mô hình nhỉnh nhất (OOF khoảng 0,930). Kiểm bằng OOF trước rồi mới tin, đúng cách làm của bài 8.4.

Gợi ý khi demo: bỏ cột `capital-gain` (thứ hai trong bảng độ quan trọng) rồi chạy lại mục 4. AUC giảm từ khoảng 0,929 xuống khoảng 0,904, tức một cột đã gánh phần lớn khoảng cách giữa boosting và logistic.